# Covid Overdose Hypothesis Project
**U.S. Drug Overdose Deaths**

**Dataset:-** `VSRR_Provisional_Drug_Overdose_Death_Counts.csv` This dataset contains historical counts of drug overdose deaths across the United States. It includes yearly and monthly data broken down by states and specific public health indicators. The dataset allows us to analyze trends over time, identify changes before and after major events, and explore national overdose patterns.

**Topic:-** Did the rate of drug overdose deaths in the United States increased significantly after the start of COVID-19 (March 2020) compared to before COVID?

**Hypothesis**
-    **H1 (alternative):** The average monthly overdose mortality rate of overdose deaths were higher after COVID-19 than before COVID-19.
-   **H0 (null):** There is no difference in the average monthly mortality rate vs. after COVID-19.





In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats


In [19]:
O_D = pd.read_csv('/Users/bk-legamo/Desktop/Portfolio-Project/Data/VSRR_Provisional_Drug_Overdose_Death_Counts.csv')


## Basic inspection

- **dimensions**
- **first few rows**
- **Column types and non null counts**
- **summary statistics for numeric columns**

In [20]:
print(O_D.shape)
print(O_D.head())
print(O_D.info())
print(O_D.describe(include='all'))

(78120, 12)
  State  Year     Month           Period        Indicator Data Value  \
0    AK  2015   January  12 month-ending  Cocaine (T40.5)        NaN   
1    AK  2015  February  12 month-ending  Cocaine (T40.5)        NaN   
2    AK  2015     March  12 month-ending  Cocaine (T40.5)        NaN   
3    AK  2015     April  12 month-ending  Cocaine (T40.5)        NaN   
4    AK  2015       May  12 month-ending  Cocaine (T40.5)        NaN   

   Percent Complete  Percent Pending Investigation State Name  \
0               100                            0.0     Alaska   
1               100                            0.0     Alaska   
2               100                            0.0     Alaska   
3               100                            0.0     Alaska   
4               100                            0.0     Alaska   

                                            Footnote Footnote Symbol  \
0  Numbers may differ from published reports usin...              **   
1  Numbers may diffe

In [21]:
print(O_D.columns.tolist())

['State', 'Year', 'Month', 'Period', 'Indicator', 'Data Value', 'Percent Complete', 'Percent Pending Investigation', 'State Name', 'Footnote', 'Footnote Symbol', 'Predicted Value']


## Data Quality Assessment

- **Checking Specific Indicator**
- **Checking for Duplicates**
- **Checking for Missing Values**

In this step, I check the dataset which include only the records related to the ‘Number of Drug Overdose Deaths’ indicator. This allows the analysis to focus specifically on overdose mortality trends rather than all other indicators contained in the dataset.also for duplicates and missing value

In [22]:
print(O_D[O_D['Indicator'] == 'Number of Drug Overdose Deaths'].head())
print(f"\nNumber of duplicated rows: {O_D.duplicated().sum()}")
O_D[O_D['Indicator'] == 'Number of Drug Overdose Deaths'].isnull().sum()

    State  Year     Month           Period                       Indicator  \
868    AK  2015   January  12 month-ending  Number of Drug Overdose Deaths   
869    AK  2015  February  12 month-ending  Number of Drug Overdose Deaths   
870    AK  2015     March  12 month-ending  Number of Drug Overdose Deaths   
871    AK  2015     April  12 month-ending  Number of Drug Overdose Deaths   
872    AK  2015       May  12 month-ending  Number of Drug Overdose Deaths   

    Data Value  Percent Complete  Percent Pending Investigation State Name  \
868        126               100                            0.0     Alaska   
869        127               100                            0.0     Alaska   
870        125               100                            0.0     Alaska   
871        126               100                            0.0     Alaska   
872        125               100                            0.0     Alaska   

                                              Footnote Footnot

State                            0
Year                             0
Month                            0
Period                           0
Indicator                        0
Data Value                       0
Percent Complete                 0
Percent Pending Investigation    0
State Name                       0
Footnote                         0
Footnote Symbol                  0
Predicted Value                  0
dtype: int64

## Data Cleaning Steps
-   **Create a date column and sort the data**

The dataset includes separate Year and Month columns.
To make time-series analysis easier, I convert these into a single date column and sort the data chronologically.

In [23]:
O_D['date'] = pd.to_datetime(O_D['Year'].astype(str) + '-' + O_D['Month'].astype(str) + '-01')
O_D = O_D.sort_values('date').reset_index(drop=True)



- **Filtering the Dataset for Overdose Death Indicators**
- **Checking Data Types**

The dataset contains multiple health indicators.
Here I extract only the rows where the indicator is “Number of Drug Overdose Deaths.” and also check the data type of each column.
This allows me to focus specifically on overdose trends and use the values.

In [24]:
D_O_D = O_D[O_D['Indicator'] == 'Number of Drug Overdose Deaths']
D_O_D.dtypes 

State                                    object
Year                                      int64
Month                                    object
Period                                   object
Indicator                                object
Data Value                               object
Percent Complete                          int64
Percent Pending Investigation           float64
State Name                               object
Footnote                                 object
Footnote Symbol                          object
Predicted Value                          object
date                             datetime64[ns]
dtype: object

In [28]:
#D_O_D['Data Value'] = D_O_D['Data Value'].str.replace(',', '', regex=False).astype(float).astype(int)

D_O_D['Data Value'] = (
    D_O_D['Data Value']
    .astype(str)
    .str.replace(',', '', regex=False)
    .astype(float)
    .astype(int)
)

D_O_D['date'] = pd.to_datetime(D_O_D['Year'].astype(str) + '-' + D_O_D['Month'].astype(str) + '-01')

D_O_D = D_O_D.sort_values('date').reset_index(drop = True)

D_O_D['monthly_deaths'] = D_O_D['Data Value'].diff()

D_O_D.dtypes

State                                    object
Year                                      int64
Month                                    object
Period                                   object
Indicator                                object
Data Value                                int64
Percent Complete                          int64
Percent Pending Investigation           float64
State Name                               object
Footnote                                 object
Footnote Symbol                          object
Predicted Value                          object
date                             datetime64[ns]
monthly_deaths                          float64
dtype: object

In [33]:
print(f"Number of missing values in 'Data Value': {D_O_D['Data Value'].isna().sum()}")

Number of missing values in 'Data Value': 0


In [35]:
D_O_D['monthly_rate'] = D_O_D['Data Value']/12
print(f"Number of missing values in 'monthly_rate': {D_O_D['monthly_rate'].isna().sum()}") 

Number of missing values in 'monthly_rate': 0


In [36]:
D_O_D.dropna(subset=['monthly_rate'])

,State,Year,Month,Period,Indicator,Data Value,Percent Complete,Percent Pending Investigation,State Name,Footnote,Footnote Symbol,Predicted Value,date,monthly_deaths,monthly_rate
0,OH,2015,January,12 month-ending,Number of Drug Overdose Deaths,2829,100,0.006038,Ohio,Numbers may differ from published reports usin...,**,"2,829",2015-01-01,NaN,235.750000
1,AR,2015,January,12 month-ending,Number of Drug Overdose Deaths,366,100,0.049797,Arkansas,Numbers may differ from published reports usin...,**,366,2015-01-01,-2463.0,30.500000
2,OK,2015,January,12 month-ending,Number of Drug Overdose Deaths,794,100,0.021171,Oklahoma,Numbers may differ from published reports usin...,**,794,2015-01-01,428.0,66.166667
3,IN,2015,January,12 month-ending,Number of Drug Overdose Deaths,1162,100,0.033776,Indiana,Numbers may differ from published reports usin...,**,"1,162",2015-01-01,368.0,96.833333
4,NY,2015,January,12 month-ending,Number of Drug Overdose Deaths,1510,100,0.278684,New York,Numbers may differ from published reports usin...,**,"1,561",2015-01-01,348.0,125.833333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6691,OK,2025,April,12 month-ending,Number of Drug Overdose Deaths,746,100,1.582328,Oklahoma,Underreported due to incomplete data.,*,971,2025-04-01,-2386.0,62.166667
6692,PR,2025,April,12 month-ending,Number of Drug Overdose Deaths,528,100,0.425930,Puerto Rico,Underreported due to incomplete data.,*,558,2025-04-01,-218.0,44.000000
6693,MT,2025,April,12 month-ending,Number of Drug Overdose Deaths,168,100,0.043433,Montana,Underreported due to incomplete data.,*,168,2025-04-01,-360.0,14.000000
6694,NY,2025,April,12 month-ending,Number of Drug Overdose Deaths,1888,100,0.374259,New York,Underreported due to incomplete data.,*,"1,977",2025-04-01,1720.0,157.333333


## Selecting US Data

- To analyze national overdose trends, I filter the dataset to only include records for the United States (State = ‘US’).

In [37]:
us_DoD = D_O_D[D_O_D['State'] == 'US']
us_DoD

,State,Year,Month,Period,Indicator,Data Value,Percent Complete,Percent Pending Investigation,State Name,Footnote,Footnote Symbol,Predicted Value,date,monthly_deaths,monthly_rate
22,US,2015,January,12 month-ending,Number of Drug Overdose Deaths,47523,100,0.143178,United States,Numbers may differ from published reports usin...,**,"48,060",2015-01-01,47249.0,3960.250000
96,US,2015,February,12 month-ending,Number of Drug Overdose Deaths,47725,100,0.146003,United States,Numbers may differ from published reports usin...,**,"48,285",2015-02-01,47519.0,3977.083333
124,US,2015,March,12 month-ending,Number of Drug Overdose Deaths,48198,100,0.145001,United States,Numbers may differ from published reports usin...,**,"48,756",2015-03-01,47326.0,4016.500000
165,US,2015,April,12 month-ending,Number of Drug Overdose Deaths,48748,100,0.146454,United States,Numbers may differ from published reports usin...,**,"49,324",2015-04-01,45886.0,4062.333333
256,US,2015,May,12 month-ending,Number of Drug Overdose Deaths,49293,100,0.146197,United States,Numbers may differ from published reports usin...,**,"49,873",2015-05-01,49175.0,4107.750000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6458,US,2024,December,12 month-ending,Number of Drug Overdose Deaths,80856,100,0.131648,United States,Underreported due to incomplete data.,*,"81,711",2024-12-01,79862.0,6738.000000
6497,US,2025,January,12 month-ending,Number of Drug Overdose Deaths,79030,100,0.157368,United States,Underreported due to incomplete data.,*,"79,909",2025-01-01,77472.0,6585.833333
6566,US,2025,February,12 month-ending,Number of Drug Overdose Deaths,77139,100,0.189803,United States,Underreported due to incomplete data.,*,"78,323",2025-02-01,75049.0,6428.250000
6593,US,2025,March,12 month-ending,Number of Drug Overdose Deaths,75669,100,0.252745,United States,Underreported due to incomplete data.,*,"77,696",2025-03-01,75236.0,6305.750000


## Before/After Split

I split the dataset into two separate periods: before March 2020 and after March 2020. This breakpoint represents the onset of the COVID-19 pandemic in the United States,By separating the data into these two intervals, I can compare monthly overdose death rates and identify any significant shifts or changes in trends following the start of the pandemic.”